<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v3/mnps_new_baseline%20v7.5.3C3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 7.5.3.C3**
> A notebook to help you get started  
> DSI DSSG + MNPS   

> # **Version 7.5.3.C3 Changes**
> - **CRITICAL FIX**: Improved justification extraction with better role patterns
> - **CRITICAL FIX**: Teacher/Librarian/Counselor/Principal/Assistant Principal/Therapist MUST have blank minor_sub_group (not I/II/III)
> - **NEW ROLE**: Added "Representative" for facility/liaison roles with HS education
> - **NEW ROLE**: Added "Assistant Principal" as distinct from Principal
> - **Instructor vs Teacher**: Check for teaching license in certifications
> - **Skilled Laborer vs Technician**: Plumbing/electrical/HVAC = Skilled Laborer
> - **Director Minor Fix**: Directors should have blank minor sub-grouping
> - **Enhanced Extraction**: More robust role extraction from justification text
> - All v7.5.3 features retained: Rate limiting, Problem Role Cheat Sheet logic

In [ ]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

In [ ]:
# ==== 2) Load data and build attribute-only view (ignore title) ====

# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")

In [ ]:
# ==== 3) Enhanced closed sets and normalization helpers - v7.5.3.C3 ====

# Get MNPS roles from the loaded data
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")

# EXPANDED closed sets - v7.5.3.C3
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Assistant Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator',
    'Skilled Laborer', 'Administrative Assistant', 'Representative'
]

MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

# CRITICAL: Roles that MUST have blank minor_sub_group (except Lead)
NO_MINOR_ROLES = ['Teacher', 'Librarian', 'Counselor', 'Principal', 'Assistant Principal', 'Therapist']

# Roles that RARELY have minor sub-grouping
RARELY_MINOR_ROLES = ['Director']

# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Director', 'Manager']

# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x) or not x or str(x).strip() == '':
        return ''
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, '')

def extract_role_from_justification(justification: str) -> tuple:
    """ENHANCED v7.5.3.C3: Extract major_role_group and minor_sub_group from justification.

    Returns: (major_role, minor_sub_group) or (None, None) if not found
    """
    if not justification:
        return None, None

    just_text = str(justification)

    # Extract major role - PRIORITY ORDER MATTERS
    major_role = None
    role_patterns = [
        # Most specific multi-word roles first
        (r"role of a[n]? ['\"]?Assistant Principal['\"]?", 'Assistant Principal'),
        (r"classified as[n]? ['\"]?Assistant Principal['\"]?", 'Assistant Principal'),
        (r"aligns with[n]? ['\"]?Assistant Principal['\"]?", 'Assistant Principal'),
        (r"\bAssistant Principal\b", 'Assistant Principal'),

        (r"role of a[n]? ['\"]?Architect \(Facility-Focused\)['\"]?", 'Architect (Facility-Focused)'),
        (r"\bArchitect \(Facility-Focused\)\b", 'Architect (Facility-Focused)'),

        (r"role of a[n]? ['\"]?Administrative Assistant['\"]?", 'Administrative Assistant'),
        (r"\bAdministrative Assistant\b", 'Administrative Assistant'),

        (r"role of a[n]? ['\"]?Clerical Support['\"]?", 'Clerical Support'),
        (r"\bClerical Support\b", 'Clerical Support'),

        (r"role of a[n]? ['\"]?Social Worker['\"]?", 'Social Worker'),
        (r"\bSocial Worker\b", 'Social Worker'),

        (r"role of a[n]? ['\"]?Skilled Laborer['\"]?", 'Skilled Laborer'),
        (r"\bSkilled Laborer\b", 'Skilled Laborer'),

        # Single-word roles - check for "role of a/an X" patterns first
        (r"role of a[n]? ['\"]?Representative['\"]?", 'Representative'),
        (r"classified as[n]? ['\"]?Representative['\"]?", 'Representative'),
        (r"\bRepresentative\b", 'Representative'),

        (r"role of a[n]? ['\"]?Coordinator['\"]?", 'Coordinator'),
        (r"classified as a[n]? ['\"]?Coordinator['\"]?", 'Coordinator'),
        (r"aligns with.*['\"]?Coordinator['\"]?", 'Coordinator'),
        (r"\bCoordinator role\b", 'Coordinator'),

        (r"role of a[n]? ['\"]?Principal['\"]?", 'Principal'),
        (r"classified as[n]? ['\"]?Principal['\"]?", 'Principal'),
        (r"aligns with.*['\"]?Principal['\"]?", 'Principal'),
        (r"\bPrincipal\b", 'Principal'),

        (r"role of a[n]? ['\"]?Director['\"]?", 'Director'),
        (r"classified as a[n]? ['\"]?Director['\"]?", 'Director'),
        (r"\bDirector\b", 'Director'),

        (r"role of a[n]? ['\"]?Manager['\"]?", 'Manager'),
        (r"classified as a[n]? ['\"]?Manager['\"]?", 'Manager'),
        (r"\bManager\b", 'Manager'),

        (r"role of a[n]? ['\"]?Supervisor['\"]?", 'Supervisor'),
        (r"\bSupervisor\b", 'Supervisor'),

        (r"role of a[n]? ['\"]?Analyst['\"]?", 'Analyst'),
        (r"classified as a[n]? ['\"]?Analyst['\"]?", 'Analyst'),
        (r"\bAnalyst\b", 'Analyst'),

        (r"role of a[n]? ['\"]?Accountant['\"]?", 'Accountant'),
        (r"\bAccountant\b", 'Accountant'),

        (r"role of a[n]? ['\"]?Technician['\"]?", 'Technician'),
        (r"\bTechnician\b", 'Technician'),

        (r"role of a[n]? ['\"]?Therapist['\"]?", 'Therapist'),
        (r"classified as a[n]? ['\"]?Therapist['\"]?", 'Therapist'),
        (r"\bTherapist\b", 'Therapist'),

        (r"role of a[n]? ['\"]?Counselor['\"]?", 'Counselor'),
        (r"\bCounselor\b", 'Counselor'),

        (r"role of a[n]? ['\"]?Librarian['\"]?", 'Librarian'),
        (r"\bLibrarian\b", 'Librarian'),

        (r"role of a[n]? ['\"]?Translator['\"]?", 'Translator'),
        (r"\bTranslator\b", 'Translator'),

        (r"role of a[n]? ['\"]?Instructor['\"]?", 'Instructor'),
        (r"\bInstructor\b", 'Instructor'),

        (r"role of a[n]? ['\"]?Teacher['\"]?", 'Teacher'),
        (r"classified as a[n]? ['\"]?Teacher['\"]?", 'Teacher'),
        (r"\bTeacher\b", 'Teacher'),

        (r"role of a[n]? ['\"]?Coach['\"]?", 'Coach'),
        (r"classified as a[n]? ['\"]?Coach['\"]?", 'Coach'),
        (r"\bCoach role\b", 'Coach'),

        (r"role of a[n]? ['\"]?Specialist['\"]?", 'Specialist'),
        (r"\bSpecialist\b", 'Specialist'),
    ]

    for pattern, role_name in role_patterns:
        if re.search(pattern, just_text, re.IGNORECASE):
            major_role = role_name
            break

    # Extract minor sub-group
    minor_sub_group = None
    minor_match = re.search(r"\b(Lead|III|II|I)\b", just_text)
    if minor_match:
        minor_sub_group = minor_match.group(1)

    return major_role, minor_sub_group

def check_teaching_license(row: pd.Series) -> bool:
    """Check if role requires teaching license/certification."""
    cert_text = str(row.get('Licenses and Certifications', '')).lower()
    return bool(re.search(r'(teaching license|teaching certification|teacher license|teacher certification|certificated)', cert_text))

def check_skilled_laborer(row: pd.Series) -> bool:
    """Check if role is skilled laborer (plumbing, electrical, carpentry, HVAC)."""
    func_text = str(row.get('Essential Functions', '')).lower()
    cert_text = str(row.get('Licenses and Certifications', '')).lower()
    combined = func_text + ' ' + cert_text
    return bool(re.search(r'(plumbing|plumber|electrical|electrician|carpentry|carpenter|hvac|welding|welder)', combined))

def check_specialist_license(row: pd.Series) -> bool:
    """Check for specialized licenses like Orientation & Mobility."""
    cert_text = str(row.get('Licenses and Certifications', '')).lower()
    return bool(re.search(r'(orientation and mobility|orientation & mobility|o&m license)', cert_text))

def force_no_minor_for_specific_roles(major_role: str, minor_role: str) -> str:
    """CRITICAL v7.5.3.C3: Force blank minor for Teacher/Librarian/Counselor/Principal/etc."""
    if major_role in NO_MINOR_ROLES:
        if minor_role == 'Lead':
            return 'Lead'
        else:
            return ''  # Force blank

    if major_role in RARELY_MINOR_ROLES:
        return ''  # Directors rarely have minor sub-grouping

    return minor_role

def discourage_specialist(text: str, proposed_major: str) -> str:
    """Enhanced logic to discourage overuse of 'Specialist'."""
    if proposed_major != 'Specialist':
        return proposed_major

    t = (text or '').lower()

    # Specialist fallback patterns
    patterns = [
        ('Technician', r'technical|repair|maintenance|install|troubleshoot|equipment'),
        ('Analyst', r'analyze|data analysis|research|evaluate|assess|statistical'),
        ('Coach', r'coach|instructional coach|plc|model lessons|co-teach|mentor'),
        ('Coordinator', r'coordinate|organize|facilitate|liaison|program coordination'),
    ]

    for major, pattern in patterns:
        if re.search(pattern, t):
            return major

    return proposed_major

print("✅ v7.5.3.C3: Enhanced role extraction and classification defined")

In [ ]:
# ==== 4) Build comprehensive KSACs text from all MNPS resources ====

def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n\n"

    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)

    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n\n"

    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)

    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"

    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)

    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"

    return ksacs_text

KSACS_TEXT = build_ksacs_text()

print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")

In [ ]:
# ==== 5) Enhanced Zero Shot Prompt - v7.5.3.C3 ====
zero_shot_prompt = \
""" Objective: Evaluate and group jobs based on similarities in job functions, not job titles.

Process:
- Compare jobs using: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, Position Summary
- Group jobs with similar functions, responsibilities, requirements, regardless of job titles
- Use MNPS standards and classifications for alignment
- Focus on actual work performed, not job titles
- **CRITICAL**: In your grouping_justification, you MUST explicitly state "This position aligns with the [ROLE NAME] role" or "classified as a [ROLE NAME]"

ROLE DISTINCTIONS:

**Technician vs Specialist vs Analyst vs Skilled Laborer:**
- Technician: Hands-on technical equipment maintenance, troubleshooting, installation
- Skilled Laborer: Trades work (plumbing, electrical, carpentry, HVAC, welding)
- Analyst: Data analysis, research, evaluation, statistical work, reporting
- Specialist: Specialized knowledge with specific licensure (use sparingly)

**Coordinator vs Coach vs Manager:**
- Coordinator: Coordination, organization, facilitation, program coordination
- Coach: Instructional support, mentoring, professional development, co-teaching
- Manager: Strategic planning, policy, budget oversight, supervision

**Instructor vs Teacher:**
- Instructor: Does NOT require teaching license (e.g., JROTC instructors)
- Teacher: Requires teaching license/certification

**Representative:**
- High school education, basic liaison/representative duties, facility coordination

**Principal vs Assistant Principal:**
- Principal: Full building leader, supervises all staff, requires Principal license
- Assistant Principal: Assists principal, limited supervision

CRITICAL MINOR SUB-GROUP RULES:
- **Teacher, Librarian, Counselor, Principal, Assistant Principal, Therapist**: NO minor sub-grouping (leave blank) unless "Lead"
- **Director**: Rarely has minor sub-grouping (usually blank)
- **III**: Very advanced KSACs, senior-level
- **II**: Intermediate complexity
- **I**: Entry-level complexity
- **Lead**: Non-executive roles leading teams

**MANDATORY**: In your grouping_justification, state explicitly: "This position aligns with the [ROLE NAME] role because..." or "classified as a [ROLE NAME]"

Output Requirements:
- major_role_group: Choose from approved MNPS roles
- minor_sub_group: Use I, II, III, Lead, or blank (MUST be blank for Teacher/Librarian/Counselor/Principal/Assistant Principal/Therapist/Director unless Lead)
- new_job_title: Incorporate major_role_group and minor_sub_group
- grouping_justification: MUST explicitly state the role classification
"""

print("✅ v7.5.3.C3: Enhanced zero shot prompt defined")

In [ ]:
# ==== 6) OpenAI API Setup with Rate Limiting Protection ====
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)

        except Exception as e:
            error_str = str(e).lower()

            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached. Error: {e}")
                    raise e
            else:
                print(f"❌ Non-rate limiting error: {e}")
                raise e

    raise Exception("Unexpected error in retry logic")

print("✅ call_llm_json_with_retry function defined")

In [ ]:
# ==== 7) Enhanced Batch Processing - v7.5.3.C3 with ALL FIXES ====
from tqdm import tqdm

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description with v7.5.3.C3 comprehensive fixes."""
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    prompt = f"""{zero_shot_prompt}

Available MNPS Roles: {', '.join(MAJOR_ALLOWED)}

{KSACS_TEXT}

Job Description to Classify:
{job_text}

**CRITICAL**:
- Ignore job title completely
- In grouping_justification, explicitly state "This position aligns with the [ROLE NAME] role" or "classified as a [ROLE NAME]"
- Teacher/Librarian/Counselor/Principal/Assistant Principal/Therapist/Director: NO minor sub-grouping (blank) unless Lead
- Check for teaching license: If yes = Teacher, if no = Instructor
- Check for trades work (plumbing/electrical/carpentry): If yes = Skilled Laborer
- Check for specialized licenses (O&M): If yes = Specialist
- Representative: High school education only, facility/liaison work

Return JSON:
{{
  "new_job_title": "Title with major and minor",
  "major_role_group": "One of approved MNPS roles",
  "minor_sub_group": "I, II, III, Lead, or blank",
  "grouping_justification": "MUST explicitly state 'This position aligns with the [ROLE] role because...'"
}}"""

    try:
        response_data = call_llm_json_with_retry(prompt, MODEL_ID)

        major_role = response_data.get('major_role_group', 'Other')
        minor_role = response_data.get('minor_sub_group', '')
        justification = response_data.get('grouping_justification', '') or ''
        new_job_title = response_data.get('new_job_title', 'Unknown')

        # CRITICAL: Extract from justification
        extracted_major, extracted_minor = extract_role_from_justification(justification)
        if extracted_major and extracted_major in MAJOR_ALLOWED:
            major_role = extracted_major
        if extracted_minor:
            minor_role = extracted_minor

        # Apply specific checks
        if check_skilled_laborer(row):
            major_role = 'Skilled Laborer'

        if check_specialist_license(row) and major_role not in ['Teacher', 'Specialist']:
            major_role = 'Specialist'

        if major_role in ['Instructor', 'Teacher']:
            if check_teaching_license(row):
                major_role = 'Teacher'
            else:
                major_role = 'Instructor'

        # Apply v7.5.3 helper
        major_role = discourage_specialist(job_text, major_role)

        # Normalize minor
        minor_role = normalize_minor(minor_role)

        # CRITICAL: Force blank minor for specific roles
        minor_role = force_no_minor_for_specific_roles(major_role, minor_role)

        # Generate title
        if not new_job_title or new_job_title == 'Unknown':
            if minor_role:
                new_job_title = f"{major_role} {minor_role}"
            else:
                new_job_title = major_role

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': new_job_title.strip(),
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'grouping_justification': justification or 'No justification provided',
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': '',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all
results = []
print("🚀 Starting v7.5.3.C3 batch processing...")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    time.sleep(0.2)

results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v753c3.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")

In [ ]:
# ==== 8) Generate Summary Statistics and Examples ====

preds = results_df.copy()

major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles',
               'specialist_count', 'no_minor_count', 'executive_lead_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['minor_sub_group'] == '').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum())
    ]
})

summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v753c3.csv"
summary_stats.to_csv(summary_path, index=False)

examples = preds[['source_row_index', 'job_title_original', 'new_job_title',
                  'major_role_group', 'minor_sub_group']].head(15)

examples_path = OUTPUTS_DIR / "examples_gpt4o_v753c3.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 v7.5.3.C3 Summary:")
print(summary_stats.to_string(index=False))
print("\n📊 Major Role Distribution:")
print(major_counts.to_string())
print("\n📊 Minor Role Distribution:")
print(minor_counts.to_string())
print("\n📋 Examples:")
print(examples.to_string(index=False))
print(f"\n✅ Saved to: {summary_path}")

In [ ]:
# ==== 9) Enhanced Quality Check - v7.5.3.C3 ====

# Check NO_MINOR_ROLES compliance
no_minor_violations = []
for idx, row in preds.iterrows():
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])

    if major_role in NO_MINOR_ROLES and minor_role and minor_role != 'Lead':
        no_minor_violations.append({
            'row_index': row['source_row_index'],
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'new_job_title': row['new_job_title']
        })

# Check extraction accuracy
extraction_mismatches = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification'])
    major_role = str(row['major_role_group'])

    extracted, _ = extract_role_from_justification(justification)
    if extracted and extracted != major_role:
        extraction_mismatches.append({
            'row_index': row['source_row_index'],
            'assigned_role': major_role,
            'extracted_from_just': extracted,
            'justification_excerpt': justification[:150]
        })

if no_minor_violations:
    no_minor_df = pd.DataFrame(no_minor_violations)
    no_minor_path = OUTPUTS_DIR / "no_minor_violations_v753c3.csv"
    no_minor_df.to_csv(no_minor_path, index=False)
    print(f"⚠️  {len(no_minor_violations)} NO_MINOR violations - saved to {no_minor_path}")
else:
    print("✅ No NO_MINOR violations")

if extraction_mismatches:
    extraction_df = pd.DataFrame(extraction_mismatches)
    extraction_path = OUTPUTS_DIR / "extraction_mismatches_v753c3.csv"
    extraction_df.to_csv(extraction_path, index=False)
    print(f"⚠️  {len(extraction_mismatches)} extraction mismatches - saved to {extraction_path}")
else:
    print("✅ All roles match justifications")

print("\n✅ v7.5.3.C3 quality check completed")